# A pretrained detector, and reading its output honestly

RetinaNet from KerasHub in three lines, then the part that matters: what the confidence threshold is actually trading, and how to choose it for your problem rather than the demo's.

**Runs on:** GPU recommended · downloads weights &nbsp;·&nbsp; **Slides:** [Chapter 12 — Object Detection](../../../course-web-slides/ch12/index.html) &nbsp;·&nbsp; **Section:** 03 — Using a pretrained detector

---

## Loading it

In [ ]:
import keras
import keras_hub
import numpy as np
import matplotlib.pyplot as plt

detector = keras_hub.models.ObjectDetector.from_preset(
    "retinanet_resnet50_fpn_coco")
print(type(detector).__name__)

Trained on COCO: 80 classes, 330,000 images. The same argument as chapters 8 and 11 — **whatever you train this week will not compete with this**, and the interesting question is what to do with it.

## Running it

In [ ]:
image_path = keras.utils.get_file(
    origin="https://img-datasets.s3.amazonaws.com/elephant.jpg")
image = np.array(keras.utils.load_img(image_path))

preds = detector.predict(image[np.newaxis, ...], verbose=0)
print({k: np.array(v).shape for k, v in preds.items()})

In [ ]:
import matplotlib.patches as patches

COCO = ["person", "bicycle", "car", "motorcycle", "airplane", "bus", "train",
        "truck", "boat", "traffic light"]   # first ten; full list in the docs

def draw(image, boxes, classes, scores, threshold=0.5, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(image)
    n = 0
    for b, c, s in zip(boxes, classes, scores):
        if s < threshold:
            continue
        n += 1
        x0, y0, x1, y1 = b
        ax.add_patch(patches.Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
                                       edgecolor="#ff7a1a", lw=2))
        ax.text(x0, y0 - 6, f"{int(c)} {s:.2f}", color="#ff7a1a",
                fontsize=9, weight="bold")
    ax.set_title(f"threshold {threshold}: {n} detections")
    ax.axis("off")
    return ax

boxes = np.array(preds["boxes"][0])
classes = np.array(preds["classes"][0])
scores = np.array(preds["confidence"][0])
draw(image, boxes, classes, scores, 0.5)
plt.show()

## The threshold is a decision, not a default

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 5))
for ax, t in zip(axes, [0.1, 0.3, 0.5, 0.8]):
    draw(image, boxes, classes, scores, t, ax=ax)
plt.tight_layout(); plt.show()

Low threshold: everything found, plus things that are not there. High threshold: only what it is sure about, and it misses.

**There is no correct value.** The right one depends on what a false positive costs against what a false negative costs, in your application — which is a business question wearing a hyperparameter's clothes.

## The precision-recall curve, which is the honest picture

In [ ]:
# With ground truth available you would sweep the threshold and plot
# precision against recall. Here is the shape of that computation.
def precision_recall(pred_boxes, pred_scores, true_boxes, iou_threshold=0.5):
    def iou(a, b):
        x0 = np.maximum(a[0], b[:, 0]); y0 = np.maximum(a[1], b[:, 1])
        x1 = np.minimum(a[2], b[:, 2]); y1 = np.minimum(a[3], b[:, 3])
        inter = np.clip(x1-x0, 0, None) * np.clip(y1-y0, 0, None)
        aa = (a[2]-a[0]) * (a[3]-a[1])
        bb = (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1])
        return inter / (aa + bb - inter + 1e-9)

    order = np.argsort(pred_scores)[::-1]
    matched = np.zeros(len(true_boxes), bool)
    tp, fp = [], []
    for i in order:
        if len(true_boxes) == 0:
            fp.append(1); tp.append(0); continue
        overlaps = iou(pred_boxes[i], true_boxes)
        j = overlaps.argmax()
        if overlaps[j] >= iou_threshold and not matched[j]:
            matched[j] = True; tp.append(1); fp.append(0)
        else:
            tp.append(0); fp.append(1)
    tp, fp = np.cumsum(tp), np.cumsum(fp)
    recall = tp / max(len(true_boxes), 1)
    precision = tp / np.maximum(tp + fp, 1)
    return precision, recall

print("Supply your own ground-truth boxes to plot this properly.")
print("mAP is the area under this curve, averaged over classes and")
print("over IoU thresholds from 0.5 to 0.95 -- which is why a single")
print("mAP number tells you much less than the curve it came from.")

## Fine-tuning on your own classes

In [ ]:
# The chapter-8 pattern, unchanged: keep the backbone, replace the head.
backbone = detector.backbone
backbone.trainable = False

print("backbone parameters:", f"{backbone.count_params():,}")
print("\nThe procedure is the one from chapter 8:")
print("  1. freeze the backbone")
print("  2. train the new detection head to convergence")
print("  3. unfreeze the top of the backbone")
print("  4. retrain both at a much lower learning rate")

One detail specific to detection: **your classes need boxes, not just labels**, and boxes are expensive to annotate. Chapter 11's SAM is the practical answer — use it to propose masks, derive boxes from them, and correct rather than draw.

## What to check before deploying a detector

- **The threshold**, chosen from your own cost of a false positive against a false negative.
- **The NMS threshold** — too aggressive and adjacent objects merge; too loose and you ship duplicates.
- **The size distribution** of your objects against the model's training distribution. A detector trained on COCO is not tuned for objects that occupy 2% of the frame.
- **What happens with zero objects.** The empty case is the one least often tested and most often encountered.
- **Latency at your batch size**, warmed up — chapter 16's lesson about timing the compiler applies here too.

---

## What to take away

- A pretrained detector is three lines and beats anything you will train this week.
- The confidence threshold is a cost decision, not a default.
- mAP hides the precision-recall curve it came from; look at the curve.
- Fine-tuning follows the chapter-8 procedure — but boxes are expensive, so consider SAM for annotation.